# PEGASUS-X Fine-Tuning with BRIO

Fine-tune `google/pegasus-x-large` on **CNN/DailyMail 3.0.0**.


**Reference**: BRIO: Bringing Order to Abstractive Summarization — Liu et al., ACL 2022

## Step 1 — Mount Google Drive

In [2]:
!rm -rf /content/drive

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Step 2 — Install Dependencies

In [3]:
!pip install -q \
    "transformers>=4.40.0" \
    "datasets>=2.18.0" \
    "evaluate>=0.4.0" \
    "rouge-score>=0.1.2" \
    "sentencepiece>=0.2.0" \
    "accelerate>=0.28.0" \
    "numpy>=1.24.0"
print('Dependencies installed.')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
Dependencies installed.


## Step 3 — Configuration

> **Edit only this cell.** Everything below runs automatically.

In [4]:
# ── Output ────────────────────────────────────────────────────────────────────
MODE       = 'brio'         # 'mle'  | 'brio'
OUTPUT_DIR = '/content/drive/MyDrive/LVTN/DATN/models'
SUBSET     = None           # None = full 287k | 300 = quick smoke-test

# ── MLE Hyperparameters ───────────────────────────────────────────────────────
EPOCHS     = 3
BATCH_SIZE = 1
GRAD_ACCUM = 32             # effective batch = BATCH_SIZE x GRAD_ACCUM
LR         = 5e-5
MAX_INPUT  = 4096           # pegasus-x-large supports up to 16384 tokens
MAX_TARGET = 128
EVAL_STEPS = 500

# ── BRIO Hyperparameters (only when MODE = 'brio') ────────────────────────────
BRIO_CANDIDATES = 8         # diverse beam candidates per article
BRIO_ALPHA      = 0.01      # contrastive loss weight
BRIO_MARGIN     = 0.001     # per-rank margin lambda
BRIO_EPOCHS     = 2
BRIO_LR         = 1e-5

# ── Misc ──────────────────────────────────────────────────────────────────────
CANDIDATES_FILE = None      # path to pre-generated candidates.jsonl (skips generation)
RESUME          = False     # resume from last checkpoint after a disconnect
NO_FP16         = False     # disable mixed precision (use if you get CUDA errors)

## Step 4 — Imports & Setup

> Do not edit below this line.

In [5]:
import json
import os
import shutil
from types import SimpleNamespace

import evaluate
import numpy as np
import torch
from datasets import load_dataset
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

cfg = SimpleNamespace(
    model_id        = 'google/pegasus-x-large',
    mode            = MODE,
    output_dir      = OUTPUT_DIR,
    subset          = SUBSET,
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    grad_accum      = GRAD_ACCUM,
    lr              = LR,
    max_input       = MAX_INPUT,
    max_target      = MAX_TARGET,
    eval_steps      = EVAL_STEPS,
    brio_candidates = BRIO_CANDIDATES,
    brio_alpha      = BRIO_ALPHA,
    brio_margin     = BRIO_MARGIN,
    brio_epochs     = BRIO_EPOCHS,
    brio_lr         = BRIO_LR,
    candidates_file = CANDIDATES_FILE,
    resume          = RESUME,
    no_fp16         = NO_FP16,
)

os.makedirs(cfg.output_dir, exist_ok=True)

print(f"{'='*55}")
print(f"  Model     : {cfg.model_id}")
print(f"  Mode      : {cfg.mode}")
print(f"  Max input : {cfg.max_input} tokens")
print(f"  Output    : {cfg.output_dir}")
print(f"  Subset    : {cfg.subset or 'full (287k)'}")
print(f"{'='*55}")

  Model     : google/pegasus-x-large
  Mode      : brio
  Max input : 4096 tokens
  Output    : /content/drive/MyDrive/LVTN/DATN/models
  Subset    : full (287k)


## Helper Functions

In [ ]:
def make_preprocess_fn(tokenizer, max_input, max_target):
    def preprocess(batch):
        model_inputs = tokenizer(
            batch['article'],
            max_length=max_input,
            truncation=True,
            padding=False,
        )
        labels = tokenizer(
            text_target=batch['highlights'],
            max_length=max_target,
            truncation=True,
            padding=False,
        )
        model_inputs['labels'] = labels['input_ids']
        return model_inputs
    return preprocess


def make_compute_metrics(tokenizer):
    rouge = evaluate.load('rouge')

    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        preds  = np.clip(preds, 0, tokenizer.vocab_size - 1)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_preds  = ['\n'.join(p.strip().split('. ')) for p in decoded_preds]
        decoded_labels = ['\n'.join(l.strip().split('. ')) for l in decoded_labels]

        scores = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
        return {k: round(v * 100, 4) for k, v in scores.items()}

    return compute_metrics


def load_and_tokenize(cfg, tokenizer):
    print('\nLoading CNN/DailyMail 3.0.0...')
    raw = load_dataset('cnn_dailymail', '3.0.0')

    if cfg.subset:
        raw['train']      = raw['train'].select(range(min(cfg.subset, len(raw['train']))))
        raw['validation'] = raw['validation'].select(range(min(max(cfg.subset // 10, 100), len(raw['validation']))))

    print(f"  Train: {len(raw['train'])}  |  Val: {len(raw['validation'])}")

    preprocess = make_preprocess_fn(tokenizer, cfg.max_input, cfg.max_target)
    cols = raw['train'].column_names
    tokenized = raw.map(preprocess, batched=True, remove_columns=cols, desc='Tokenizing')
    tokenized.set_format('torch')
    return raw, tokenized


def build_training_args(cfg, output_dir, epochs, lr):
    return Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum,
        learning_rate=lr,
        warmup_ratio=0.06,
        weight_decay=0.01,
        max_grad_norm=0.1,
        label_smoothing_factor=0.1,
        fp16=not cfg.no_fp16,
        gradient_checkpointing=True,
        predict_with_generate=True,
        generation_max_length=cfg.max_target,
        generation_num_beams=4,
        eval_strategy='steps',
        eval_steps=cfg.eval_steps,
        save_strategy='steps',
        save_steps=cfg.eval_steps,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model='rouge2',
        greater_is_better=True,
        logging_steps=50,
        logging_first_step=True,
        report_to='none',
    )


print('Helper functions defined.')

Helper functions defined.


## Load Tokenizer, Model & Dataset

Downloads `google/pegasus-x-large` (~2.3 GB). Cached after first run.

In [7]:
print(f'\nLoading tokenizer & model from {cfg.model_id}...')
tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)
model     = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_id)
print(f'  Parameters: {model.num_parameters() / 1e6:.1f}M')

raw, tokenized = load_and_tokenize(cfg, tokenizer)


Loading tokenizer & model from google/pegasus-x-large...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

  Train: 287113  |  Val: 13368


Tokenizing:   0%|          | 0/287113 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/13368 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/11490 [00:00<?, ? examples/s]

## Phase 1 — MLE Fine-Tuning

Standard cross-entropy training on CNN/DailyMail gold summaries.  
Always runs first, even in `brio` mode.  
**Estimated time on A100**: ~10–12 h for full 287k dataset, 3 epochs.

In [ ]:
def run_mle(cfg, model, tokenizer, tokenized):
    print('\n── Phase 1: MLE fine-tuning ──────────────────────────────')
    mle_dir = os.path.join(cfg.output_dir, 'mle')
    training_args = build_training_args(cfg, mle_dir, cfg.epochs, cfg.lr)

    collator = DataCollatorForSeq2Seq(
        tokenizer, model=model, padding=True,
        pad_to_multiple_of=8 if not cfg.no_fp16 else None,
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['validation'],
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=make_compute_metrics(tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    checkpoint = mle_dir if cfg.resume and os.path.isdir(mle_dir) else None
    trainer.train(resume_from_checkpoint=checkpoint)
    trainer.save_model(mle_dir)
    tokenizer.save_pretrained(mle_dir)
    print(f'  MLE model saved -> {mle_dir}')
    return mle_dir


mle_dir = run_mle(cfg, model, tokenizer, tokenized)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



── Phase 1: MLE fine-tuning ──────────────────────────────


TypeError: Seq2SeqTrainer.__init__() got an unexpected keyword argument 'tokenizer'

## Phase 2a — Candidate Generation *(BRIO only)*

Skipped automatically when `MODE = 'mle'`.

Generates `BRIO_CANDIDATES` diverse beam-search summaries per article using the MLE model,  
scores each with ROUGE against the gold summary, and writes results to `candidates.jsonl`.  
If the file already exists (e.g. after a disconnect), generation is skipped.

In [ ]:
def rouge_score_single(pred, ref, scorer):
    s = scorer.score(ref, pred)
    return (s['rouge1'].fmeasure + s['rouge2'].fmeasure + s['rougeL'].fmeasure) / 3


def generate_candidates(cfg, model_dir, raw_train, tokenizer):
    from rouge_score import rouge_scorer as rs_lib

    out_path = os.path.join(cfg.output_dir, 'candidates.jsonl')
    if os.path.exists(out_path):
        print(f'  Candidates already exist: {out_path} — skipping generation')
        return out_path

    print(f'\n── Phase 2a: Generating {cfg.brio_candidates} candidates per article ──')
    print(f'  Loading MLE model from {model_dir}...')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    gen_model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(device)
    gen_model.eval()
    scorer = rs_lib.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    N          = cfg.brio_candidates
    num_beams  = max(N, 8)
    num_groups = min(4, N)

    written = 0
    with open(out_path, 'w') as f:
        for idx, example in enumerate(raw_train):
            if idx % 500 == 0:
                print(f'  {idx}/{len(raw_train)}')

            inputs = tokenizer(
                example['article'],
                max_length=cfg.max_input,
                truncation=True,
                return_tensors='pt',
            ).to(device)

            with torch.no_grad():
                outputs = gen_model.generate(
                    **inputs,
                    num_beams=num_beams,
                    num_beam_groups=num_groups,
                    diversity_penalty=0.8,
                    num_return_sequences=N,
                    max_length=cfg.max_target,
                    early_stopping=True,
                )

            candidates = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            scores     = [rouge_score_single(c, example['highlights'], scorer) for c in candidates]

            f.write(json.dumps({
                'article':    example['article'],
                'gold':       example['highlights'],
                'candidates': candidates,
                'scores':     scores,
            }) + '\n')
            written += 1

    print(f'  Wrote {written} examples -> {out_path}')
    del gen_model
    torch.cuda.empty_cache()
    return out_path


if cfg.mode == 'brio':
    candidates_file = cfg.candidates_file or generate_candidates(cfg, mle_dir, raw['train'], tokenizer)
    print(f'Candidates file: {candidates_file}')
else:
    candidates_file = None
    print('MODE = mle — skipping candidate generation.')

## Phase 2b — BRIO Contrastive Training *(BRIO only)*

Skipped automatically when `MODE = 'mle'`.

Adds a **margin-ranking loss** on top of MLE so the model assigns higher probability to candidates  
that scored higher on ROUGE. This directly aligns training with beam-search inference.

```
total_loss = L_MLE(gold) + alpha × L_contrast(candidates)
L_ij       = max(0, NLL[i] − NLL[j] + margin × (j−i))   # i better than j
```

**Paper**: BRIO: Bringing Order to Abstractive Summarization — Liu et al., ACL 2022

In [ ]:
class BRIOTrainer(Seq2SeqTrainer):
    def __init__(self, *args, candidates_ds, alpha=0.01, margin=0.001, **kwargs):
        super().__init__(*args, **kwargs)
        self._cands    = {row['article']: row for row in candidates_ds}
        self.alpha     = alpha
        self.margin    = margin
        self._loss_fct = CrossEntropyLoss(reduction='none', ignore_index=-100)

    def _seq_nll(self, model, input_ids, attention_mask, cand_ids, cand_mask):
        labels = cand_ids.clone()
        labels[cand_mask == 0] = -100

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        logits = outputs.logits
        M, T, V = logits.shape

        token_loss = self._loss_fct(
            logits.reshape(M * T, V),
            labels.reshape(M * T),
        ).reshape(M, T)

        valid = (labels != -100).float()
        nll   = (token_loss * valid).sum(-1) / valid.sum(-1).clamp(min=1)
        return nll

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        mle_out  = super().compute_loss(model, inputs, return_outputs=True)
        mle_loss = mle_out[0] if isinstance(mle_out, tuple) else mle_out

        if self.alpha == 0 or not self._cands:
            return (mle_loss, mle_out[1]) if return_outputs else mle_loss

        src_ids  = inputs.get('input_ids')
        articles = self.tokenizer.batch_decode(src_ids, skip_special_tokens=True)

        batch_cands, batch_scores, valid_indices = [], [], []
        for b_idx, article in enumerate(articles):
            row = self._cands.get(article)
            if row is None:
                continue
            batch_cands.append(row['candidates'])
            batch_scores.append(row['scores'])
            valid_indices.append(b_idx)

        if not valid_indices:
            return (mle_loss, mle_out[1]) if return_outputs else mle_loss

        device  = src_ids.device
        N       = len(batch_cands[0])
        B_valid = len(valid_indices)

        flat_cands = [c for cands in batch_cands for c in cands]
        enc = self.tokenizer(
            flat_cands,
            max_length=self.args.generation_max_length,
            truncation=True,
            padding=True,
            return_tensors='pt',
        ).to(device)

        valid_src_ids  = src_ids[valid_indices]
        valid_src_mask = inputs['attention_mask'][valid_indices]
        rep_src_ids    = valid_src_ids.unsqueeze(1).expand(-1, N, -1).reshape(B_valid * N, -1)
        rep_src_mask   = valid_src_mask.unsqueeze(1).expand(-1, N, -1).reshape(B_valid * N, -1)

        nlls = self._seq_nll(model, rep_src_ids, rep_src_mask, enc['input_ids'], enc['attention_mask'])
        nlls = nlls.reshape(B_valid, N)

        scores_t    = torch.tensor(batch_scores, device=device)
        order       = scores_t.argsort(dim=-1, descending=True)
        sorted_nlls = nlls.gather(1, order)

        contrast_loss = torch.tensor(0.0, device=device, requires_grad=True)
        count = 0
        for i in range(N):
            for j in range(i + 1, N):
                pair_loss     = torch.clamp(sorted_nlls[:, i] - sorted_nlls[:, j] + self.margin * (j - i), min=0)
                contrast_loss = contrast_loss + pair_loss.mean()
                count += 1

        if count:
            contrast_loss = contrast_loss / count

        total_loss = mle_loss + self.alpha * contrast_loss
        return (total_loss, mle_out[1]) if return_outputs else total_loss


def run_brio(cfg, mle_dir, tokenizer, tokenized, candidates_file):
    print('\n── Phase 2b: BRIO contrastive fine-tuning ────────────────')

    rows = []
    with open(candidates_file) as f:
        for line in f:
            rows.append(json.loads(line))
    print(f'  {len(rows)} candidate sets loaded')

    brio_model    = AutoModelForSeq2SeqLM.from_pretrained(mle_dir)
    brio_dir      = os.path.join(cfg.output_dir, 'brio')
    training_args = build_training_args(cfg, brio_dir, cfg.brio_epochs, cfg.brio_lr)

    collator = DataCollatorForSeq2Seq(
        tokenizer, model=brio_model, padding=True,
        pad_to_multiple_of=8 if not cfg.no_fp16 else None,
    )
    trainer = BRIOTrainer(
        model=brio_model,
        args=training_args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['validation'],
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=make_compute_metrics(tokenizer),
        candidates_ds=rows,
        alpha=cfg.brio_alpha,
        margin=cfg.brio_margin,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()
    trainer.save_model(brio_dir)
    tokenizer.save_pretrained(brio_dir)
    print(f'  BRIO model saved -> {brio_dir}')
    return brio_dir


if cfg.mode == 'brio':
    final_dir = run_brio(cfg, mle_dir, tokenizer, tokenized, candidates_file)
else:
    final_dir = mle_dir
    print('MODE = mle — skipping BRIO stage.')

## Copy Final Model to Output Root

The AI service loads from `OUTPUT_DIR` directly. This copies the best checkpoint  
from `mle/` or `brio/` subdirectory up to the root output folder.

In [ ]:
print(f'\nCopying final model from {final_dir} to {cfg.output_dir}...')
for fname in os.listdir(final_dir):
    src = os.path.join(final_dir, fname)
    dst = os.path.join(cfg.output_dir, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
print(f'Done. Model ready at: {cfg.output_dir}')

## Final Evaluation

Evaluates the final model on the CNN/DailyMail **test set** (first 500 examples) and reports ROUGE scores.

In [ ]:
def final_eval(model_dir, cfg, tokenizer, raw):
    print('\n── Final evaluation on test set (n=500) ──────────────────')
    eval_model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
    test_sub   = raw['test'].select(range(min(500, len(raw['test']))))
    preprocess = make_preprocess_fn(tokenizer, cfg.max_input, cfg.max_target)
    test_tok   = test_sub.map(preprocess, batched=True, remove_columns=test_sub.column_names, desc='Tokenizing test')
    test_tok.set_format('torch')

    eval_args = Seq2SeqTrainingArguments(
        output_dir='/tmp/eval_tmp',
        per_device_eval_batch_size=cfg.batch_size,
        predict_with_generate=True,
        generation_max_length=cfg.max_target,
        generation_num_beams=4,
        fp16=not cfg.no_fp16,
        report_to='none',
    )
    collator = DataCollatorForSeq2Seq(tokenizer, model=eval_model, padding=True)
    trainer  = Seq2SeqTrainer(
        model=eval_model, args=eval_args, tokenizer=tokenizer,
        data_collator=collator, compute_metrics=make_compute_metrics(tokenizer),
    )

    results = trainer.evaluate(test_tok, metric_key_prefix='test')
    print('\n' + '=' * 55)
    print('  Test Results')
    print('=' * 55)
    for k, v in sorted(results.items()):
        if 'rouge' in k:
            print(f'  {k:<35} {v:.4f}')
    print('=' * 55)
    return results


results = final_eval(cfg.output_dir, cfg, tokenizer, raw)